# 05 PPO：标准 Actor-Critic、GAE 与 PPO-Clip

本节自行实现标准在线 PPO 教学基线。Actor 生成回答，Critic 估计每个 Response Token 的状态价值，Old Policy 保存行为策略 Log Probability，确定性 Verifier 提供 Outcome Reward。PPO 与 GRPO 分开：PPO 使用 Critic 和 GAE，不依赖组内相对奖励。

DGX Spark 阶段只运行 CPU 数值 Smoke Test；不加载 Qwen3、不启动 vLLM、不进行 GPU Rollout 或更新。真实实验必须在 RTX 5090 官方训练/vLLM Profile 中同步串行执行。

## 学习目标

- 对齐 Prompt、Completion、EOS、Padding 和 Response Mask；
- 从 `r_t + gamma V_{t+1} - V_t` 推导 GAE 和 Return；
- 实现 Probability Ratio、PPO-Clip、Value Loss、Entropy 和 KL；
- 使用共享 Backbone + Value Head 的轻量 Actor-Critic 结构；
- 记录行为策略版本、Reward、Advantage、Clip Fraction 和 Explained Variance。

## 1. PPO 训练协议

同步串行流程为：`Actor Rollout → 保存 Token/Logprob/Mask → Verifier Reward → Critic Value → GAE → PPO Update → 保存 Adapter`。vLLM 和训练模型不同时驻留 GPU。Gravity 通常只作为能力保持评测，主 PPO 任务是 Equation Numeric 和 Bit Manipulation。

核心目标：

\[
r_t(\theta)=\exp(\log\pi_\theta-\log\pi_{old}),\quad
L_{clip}=-E[\min(r_t A_t,\operatorname{clip}(r_t,1-\epsilon,1+\epsilon)A_t)]
\]

\[
\delta_t=r_t+\gamma(1-d_t)V_{t+1}-V_t,\quad
A_t=\delta_t+\gamma\lambda(1-d_t)A_{t+1}
\]

In [ ]:
from __future__ import annotations

import json
import math
import os
import time
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

PROJECT_ROOT = Path.cwd()
RESULT_DIR = PROJECT_ROOT / "data" / "processed" / "ppo"
CHECKPOINT_ROOT = PROJECT_ROOT / "data" / "processed" / "checkpoints"
SFT_MANIFEST_PATH = CHECKPOINT_ROOT / "sft_manifest.json"
TRAIN_PATH = PROJECT_ROOT / "data" / "processed" / "splits" / "train.csv"
MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"
BASE_MODEL_PATH = os.environ.get("QWEN3_MODEL_PATH", MODEL_ID)
SEED = 42
RUN_PPO = os.environ.get("RUN_PPO", "0") == "1"  # umbrella flag
RUN_PPO_ROLLOUT = os.environ.get("RUN_PPO_ROLLOUT", "1" if RUN_PPO else "0") == "1"
RUN_PPO_TRAINING = os.environ.get("RUN_PPO_TRAINING", "1" if RUN_PPO else "0") == "1"
RUN_MODE = os.environ.get("RUN_MODE", "smoke")
GROUP_TASKS = ("equation_numeric", "bit_manipulation")
SMOKE_PROMPTS_PER_TASK = 2
MAX_NEW_TOKENS = 2048
# T=1/top-p=1 makes the behavior distribution explicit for the first alignment test.
TEMPERATURE = 1.0
TOP_P = 1.0
TRAINING_PROFILE = "official-pt-rl-cuda"
GAMMA = 0.99
GAE_LAMBDA = 0.95
PPO_CLIP_EPS = 0.2
VALUE_COEF = 0.5
ENTROPY_COEF = 0.01
KL_COEF = 0.0
LEARNING_RATE = 1e-5
PPO_ROLLOUT_PATH = RESULT_DIR / "ppo_smoke_rollouts.jsonl"
PPO_ROLLOUT_MANIFEST_PATH = RESULT_DIR / "ppo_rollout_manifest.json"
PPO_TRAINING_MANIFEST_PATH = RESULT_DIR / "ppo_training_manifest.json"

np.random.seed(SEED)
torch.manual_seed(SEED)
print({"model_id": MODEL_ID, "run_ppo": RUN_PPO, "run_mode": RUN_MODE, "max_new_tokens": MAX_NEW_TOKENS, "tasks": GROUP_TASKS})

## 2. Rollout 数据结构与 Mask 约定

`input_ids` 包含 Prompt 和 Completion；`response_mask=1` 只覆盖 Completion Token，EOS 属于 Completion，Padding 为 0。`old_logprobs` 和 `old_values` 必须来自生成这批 Rollout 的行为策略版本，不能用更新后的 Policy 重新计算后冒充 Old Policy。

In [ ]:
@dataclass
class PPORollout:
    input_ids: torch.Tensor
    attention_mask: torch.Tensor
    response_mask: torch.Tensor
    old_logprobs: torch.Tensor
    old_values: torch.Tensor
    rewards: torch.Tensor
    dones: torch.Tensor
    policy_version: str
    verifier_version: str
    finish_reason: str
    truncated: bool
    task_type: str
    sample_id: str


def validate_rollout_alignment(rollout: PPORollout) -> None:
    shapes = [rollout.input_ids.shape, rollout.attention_mask.shape, rollout.response_mask.shape, rollout.old_logprobs.shape, rollout.old_values.shape, rollout.rewards.shape, rollout.dones.shape]
    if len({tuple(shape) for shape in shapes}) != 1:
        raise ValueError(f"rollout tensors are misaligned: {shapes}")
    if rollout.input_ids.ndim != 2:
        raise ValueError("rollout tensors must have shape [B,T]")
    if not torch.all((rollout.response_mask == 0) | (rollout.response_mask == 1)):
        raise ValueError("response_mask must be binary")
    if not torch.all((rollout.attention_mask == 0) | (rollout.attention_mask == 1)):
        raise ValueError("attention_mask must be binary")
    if not torch.all(rollout.response_mask <= rollout.attention_mask):
        raise ValueError("response_mask cannot include padding")
    if not torch.all(rollout.dones <= rollout.response_mask):
        raise ValueError("done can only occur on a response token")
    if torch.any(rollout.dones.sum(dim=1) > 1):
        raise ValueError("each sequence can have at most one observed EOS")
    if rollout.truncated and torch.any(rollout.dones > 0):
        raise ValueError("a length-truncated rollout cannot claim an observed EOS")


def outcome_reward(correct: bool | torch.Tensor, response_mask: torch.Tensor, task_type: str) -> torch.Tensor:
    """Put the binary outcome reward on each row's final valid Response token."""
    if response_mask.ndim == 1:
        response_mask = response_mask.unsqueeze(0)
    reward = torch.zeros_like(response_mask, dtype=torch.float32)
    correct_values = torch.as_tensor(correct, dtype=torch.bool, device=response_mask.device).reshape(-1)
    if correct_values.numel() == 1 and response_mask.size(0) > 1:
        correct_values = correct_values.expand(response_mask.size(0))
    if correct_values.numel() != response_mask.size(0):
        raise ValueError("correct must have one value per rollout row")
    for row_index in range(response_mask.size(0)):
        active_positions = torch.nonzero(response_mask[row_index].bool(), as_tuple=False).flatten()
        if active_positions.numel():
            reward[row_index, int(active_positions[-1])] = float(correct_values[row_index])
    return reward


def build_cpu_rollout(batch_size: int = 2, time_steps: int = 7, vocab_size: int = 32) -> PPORollout:
    torch.manual_seed(SEED)
    input_ids = torch.randint(0, vocab_size, (batch_size, time_steps))
    attention_mask = torch.ones(batch_size, time_steps, dtype=torch.long)
    response_mask = torch.zeros(batch_size, time_steps, dtype=torch.long)
    response_mask[:, 2:] = 1
    old_logprobs = torch.randn(batch_size, time_steps) * 0.1
    old_values = torch.randn(batch_size, time_steps) * 0.1
    rewards = torch.zeros(batch_size, time_steps)
    rewards[:, -1] = torch.tensor([1.0, 0.0])
    dones = torch.zeros(batch_size, time_steps)
    dones[:, -1] = 1.0
    rollout = PPORollout(input_ids, attention_mask, response_mask, old_logprobs, old_values, rewards, dones, "cpu_fixture", "toy_verifier", "stop", False, "equation_numeric", "cpu-ppo")
    validate_rollout_alignment(rollout)
    return rollout

## 3. GAE、Return、PPO-Clip 和诊断指标

Padding 与终止状态都通过 Mask 处理。终止位置的 bootstrap 被关闭；Padding 不贡献 Advantage、Value Loss、Entropy 或 KL。`advantages` 只在有效 Response Token 上标准化。

In [ ]:
def masked_mean(values: torch.Tensor, mask: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    mask = mask.to(values.dtype)
    return (values * mask).sum() / mask.sum().clamp_min(eps)


def compute_gae(rewards: torch.Tensor, values: torch.Tensor, dones: torch.Tensor, response_mask: torch.Tensor, gamma: float = GAMMA, lam: float = GAE_LAMBDA) -> tuple[torch.Tensor, torch.Tensor]:
    if rewards.shape != values.shape or rewards.shape != dones.shape or rewards.shape != response_mask.shape:
        raise ValueError("rewards, values, dones and response_mask must share [B,T]")
    batch, time_steps = rewards.shape
    mask = response_mask.to(rewards.dtype)
    advantages = torch.zeros_like(rewards)
    next_advantage = torch.zeros(batch, device=rewards.device, dtype=rewards.dtype)
    for t in reversed(range(time_steps)):
        valid = mask[:, t]
        next_valid = mask[:, t + 1] if t + 1 < time_steps else torch.zeros_like(valid)
        nonterminal = (1.0 - dones[:, t].to(rewards.dtype)) * next_valid
        next_value = values[:, t + 1] if t + 1 < time_steps else torch.zeros_like(values[:, t])
        delta = (rewards[:, t] + gamma * next_value * nonterminal - values[:, t]) * valid
        next_advantage = delta + gamma * lam * nonterminal * next_advantage
        next_advantage = next_advantage * valid
        advantages[:, t] = next_advantage
    returns = (advantages + values) * mask
    return advantages, returns


def normalize_masked(values: torch.Tensor, mask: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    mean = masked_mean(values, mask)
    variance = masked_mean((values - mean) ** 2, mask)
    return ((values - mean) / torch.sqrt(variance + eps)) * mask.to(values.dtype)


def ppo_clip_loss(old_logprobs: torch.Tensor, new_logprobs: torch.Tensor, advantages: torch.Tensor, response_mask: torch.Tensor, clip_eps: float = PPO_CLIP_EPS) -> dict[str, torch.Tensor]:
    ratio = torch.exp((new_logprobs - old_logprobs).clamp(-20, 20))
    unclipped = ratio * advantages
    clipped_ratio = ratio.clamp(1.0 - clip_eps, 1.0 + clip_eps)
    clipped = clipped_ratio * advantages
    objective = torch.minimum(unclipped, clipped)
    clip_fraction = ((ratio - clipped_ratio).abs() > 1e-8).to(torch.float32)
    return {
        "policy_loss": -masked_mean(objective, response_mask),
        "ratio": ratio.detach(),
        "clip_fraction": masked_mean(clip_fraction, response_mask),
    }


def value_loss(values: torch.Tensor, returns: torch.Tensor, response_mask: torch.Tensor) -> torch.Tensor:
    return masked_mean((values - returns.detach()) ** 2, response_mask)


def entropy_from_logits(logits: torch.Tensor, response_mask: torch.Tensor) -> torch.Tensor:
    entropy = -(F.softmax(logits, dim=-1) * F.log_softmax(logits, dim=-1)).sum(dim=-1)
    return masked_mean(entropy, response_mask)


def approx_kl(old_logprobs: torch.Tensor, new_logprobs: torch.Tensor, response_mask: torch.Tensor) -> torch.Tensor:
    return masked_mean(old_logprobs - new_logprobs, response_mask)


def explained_variance(values: torch.Tensor, returns: torch.Tensor, response_mask: torch.Tensor) -> torch.Tensor:
    mask = response_mask.bool()
    y, y_pred = returns.detach()[mask], values.detach()[mask]
    if y.numel() < 2 or torch.var(y) < 1e-8:
        return torch.tensor(0.0, device=values.device)
    return 1.0 - torch.var(y - y_pred) / torch.var(y)

## 4. 共享 Backbone + Scalar Value Head

真实 Qwen3 模型由 Transformers 加载，Value Head 读取最后一层 Hidden State；Policy Logits 仍来自 Causal LM Head。LoRA / QLoRA 的 Adapter 只更新 Actor Backbone 的可训练参数，Value Head 是否共享需通过显存和稳定性实验确认。这里的 Tiny 模型用于 CPU 机制测试。

**时间轴约定（本轮审查修订）**：隐藏状态位置 `t` 是预测 token `t+1` 的动作前状态；因此 Policy Logprob 使用 `logits[:, :-1]`，Value 使用 `values[:, :-1]`，Reward、Done 和 Response Mask 使用 token 位置 `[:, 1:]`。Padding、Prompt 和截断后的不可见状态不参与 GAE 与 Value Loss。


In [ ]:
class TinyActorCritic(nn.Module):
    def __init__(self, vocab_size: int = 32, hidden_size: int = 24):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.trunk = nn.Sequential(nn.Linear(hidden_size, hidden_size), nn.Tanh())
        self.policy_head = nn.Linear(hidden_size, vocab_size)
        self.value_head = nn.Linear(hidden_size, 1)
    def forward(self, input_ids, attention_mask=None):
        hidden = self.trunk(self.embedding(input_ids))
        return {"logits": self.policy_head(hidden), "values": self.value_head(hidden).squeeze(-1)}


class CausalLMValueHead(nn.Module):
    """Value at full-sequence position t estimates the state before token t+1.

    The scalar head is deliberately colocated with the backbone and kept in
    FP32. Hidden states are cast to the head dtype before the matmul, so a
    BF16/NF4 base model never receives a mismatched-device or mismatched-dtype
    linear operation.
    """
    def __init__(self, base_model):
        super().__init__()
        self.base_model = base_model
        hidden_size = int(base_model.config.hidden_size)
        base_parameter = next(base_model.parameters())
        self.value_head = nn.Linear(hidden_size, 1, bias=False).to(device=base_parameter.device, dtype=torch.float32)
    def forward(self, input_ids, attention_mask=None):
        outputs = self.base_model(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True, use_cache=False)
        hidden = outputs.hidden_states[-1].to(device=self.value_head.weight.device, dtype=self.value_head.weight.dtype)
        return {"logits": outputs.logits, "values": self.value_head(hidden).squeeze(-1)}


def token_logprobs(logits: torch.Tensor, input_ids: torch.Tensor) -> torch.Tensor:
    log_probs = F.log_softmax(logits[:, :-1, :], dim=-1)
    return log_probs.gather(-1, input_ids[:, 1:].unsqueeze(-1)).squeeze(-1)


def align_response_mask(response_mask: torch.Tensor) -> torch.Tensor:
    # The token at position t is predicted by logits at position t-1.
    return response_mask[:, 1:].to(torch.float32)


def align_ppo_action_axis(
    rewards: torch.Tensor,
    old_values: torch.Tensor,
    dones: torch.Tensor,
    response_mask: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    """Align all rollout fields to actions predicting token positions 1..T-1."""
    shapes = {tuple(value.shape) for value in (rewards, old_values, dones, response_mask)}
    if len(shapes) != 1 or rewards.ndim != 2 or rewards.size(1) < 2:
        raise ValueError("PPO full-sequence tensors must share [B,T] with T>=2")
    # Hidden state t predicts token t+1; reward/done/mask live on token t+1.
    return rewards[:, 1:], old_values[:, :-1], dones[:, 1:], response_mask[:, 1:].to(torch.float32)


## 5. 一次 PPO Update

Old Log Probability 和 Old Value 由 Rollout 固定；更新时只重新计算当前 Actor/Critic。总 Loss 为 Policy Loss + Value Loss − Entropy；本实现把 KL 记录为诊断，`KL_COEF=0` 时不重复惩罚。

In [ ]:
def ppo_update(model: nn.Module, optimizer: torch.optim.Optimizer, rollout: PPORollout, clip_eps: float = PPO_CLIP_EPS, value_coef: float = VALUE_COEF, entropy_coef: float = ENTROPY_COEF, kl_coef: float = KL_COEF) -> dict[str, float]:
    validate_rollout_alignment(rollout)
    outputs = model(rollout.input_ids, rollout.attention_mask)
    logits, values = outputs["logits"], outputs["values"]
    new_logprobs = token_logprobs(logits, rollout.input_ids)  # [B,T-1], actions for token positions 1..T-1
    old_logprobs = rollout.old_logprobs[:, 1:]
    action_rewards, action_old_values, action_dones, action_mask = align_ppo_action_axis(
        rollout.rewards, rollout.old_values, rollout.dones, rollout.response_mask
    )
    advantages, returns = compute_gae(action_rewards, action_old_values, action_dones, action_mask)
    advantages = normalize_masked(advantages, action_mask)
    policy = ppo_clip_loss(old_logprobs, new_logprobs, advantages, action_mask, clip_eps)
    # The value at hidden position t estimates the state before action token t+1.
    current_action_values = values[:, :-1]
    value = value_loss(current_action_values, returns, action_mask)
    entropy = entropy_from_logits(logits[:, :-1, :], action_mask)
    kl = approx_kl(old_logprobs, new_logprobs, action_mask)
    total = policy["policy_loss"] + value_coef * value - entropy_coef * entropy + kl_coef * kl
    optimizer.zero_grad(set_to_none=True)
    total.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    return {
        "total_loss": float(total.detach()),
        "policy_loss": float(policy["policy_loss"].detach()),
        "value_loss": float(value.detach()),
        "entropy": float(entropy.detach()),
        "approx_kl": float(kl.detach()),
        "clip_fraction": float(policy["clip_fraction"].detach()),
        "advantage_mean": float(masked_mean(advantages, action_mask).detach()),
        "advantage_std": float(torch.sqrt(masked_mean(advantages ** 2, action_mask)).detach()),
        "explained_variance": float(explained_variance(current_action_values, returns, action_mask).detach()),
        "action_tokens": float(action_mask.sum().detach()),
    }

## 6. GPU Rollout / PPO 入口（当前关闭）

真实入口使用 vLLM 生成并保留行为策略 Logprobs；如果 vLLM Profile 无法返回所需 Logprobs，必须停止并记录兼容性问题，不能用当前 Policy Logprobs 冒充 Old Policy。训练完成后保存 Adapter，再串行切回 vLLM。

### 6.1 读取并校验 SFT Manifest

In [ ]:
def read_sft_manifest(path: Path = SFT_MANIFEST_PATH) -> dict[str, object]:
    if not path.exists():
        raise FileNotFoundError(f"SFT Manifest 不存在：{path}；请先运行 Notebook 03")
    manifest = json.loads(path.read_text(encoding="utf-8"))
    if manifest.get("schema_version") != "nemotron-sft-checkpoint/v1" or manifest.get("base_model", {}).get("id") != MODEL_ID:
        raise ValueError("SFT Manifest schema 或 Base Model 不匹配")
    if manifest.get("status") != "completed":
        raise ValueError("SFT Manifest 尚未标记为 completed，不能初始化 PPO")
    adapter_path = Path(resolve_manifest_path(str(manifest.get("adapter_path", ""))))
    if not adapter_path.is_dir() or not (adapter_path / "adapter_config.json").exists():
        raise FileNotFoundError(f"SFT Adapter 不存在或缺少 adapter_config.json：{adapter_path}")
    return manifest


def resolve_manifest_path(value: str) -> str:
    path = Path(value)
    return str(path if path.is_absolute() else PROJECT_ROOT / path)


### 6.2 解析 vLLM 行为策略 Logprob与任务答案验证

In [ ]:
def _chosen_logprob_values(generated) -> list[float]:
    raw, token_ids = generated.logprobs, list(generated.token_ids or [])
    if raw is None or len(raw) != len(token_ids): raise RuntimeError("vLLM Logprob/Token ID 长度不一致")
    values = []
    for token_id, candidates in zip(token_ids, raw):
        selected = candidates.get(token_id, candidates.get(str(token_id))) if isinstance(candidates, dict) else candidates
        if selected is None and isinstance(candidates, dict) and len(candidates) == 1: selected = next(iter(candidates.values()))
        value = getattr(selected, "logprob", selected)
        if value is None or not math.isfinite(float(value)): raise RuntimeError("vLLM 行为 Logprob 非有限")
        values.append(float(value))
    return values


OFFICIAL_VERIFIER_VERSION = "official_nvidia_metric_v2"


def _normalize_answer_text(value: object) -> str:
    text = str(value).strip()
    return __import__("re").sub(r"^```(?:text|json)?\s*|\s*```$", "", text, flags=__import__("re").I).strip()


def _extract_rollout_answer(text: str, task_type: str) -> tuple[str | None, str]:
    """Notebook 01-compatible task-aware extraction, with markers first."""
    import re
    text = str(text)
    boxed = re.findall(r"\\boxed\{([^{}]+)\}", text)
    if boxed:
        return _normalize_answer_text(boxed[-1]), "boxed"
    tagged = re.findall(r"<(?:final|answer)>\s*(.*?)\s*</(?:final|answer)>", text, flags=re.I | re.S)
    if tagged:
        return _normalize_answer_text(tagged[-1]), "tagged"
    if task_type == "bit_manipulation":
        matches = re.findall(r"(?<![01])[01]{8}(?![01])", text)
        if matches:
            return matches[-1], "binary_fallback"
    if task_type in {"gravity", "equation_numeric"}:
        matches = re.findall(r"(?<![A-Za-z])[-+]?\d+(?:\.\d+)?", text)
        if matches:
            return matches[-1], "numeric_fallback"
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    return (_normalize_answer_text(lines[-1]), "last_nonempty_line") if lines else (None, "empty")


def _verify_rollout(task_type: str, completion: str, expected: str, truncated: bool = False) -> tuple[bool, str]:
    if truncated:
        return False, "truncated_output"
    predicted, _ = _extract_rollout_answer(completion, task_type)
    if predicted is None:
        return False, "not_extracted"
    predicted_text, expected_text = _normalize_answer_text(predicted), _normalize_answer_text(expected)
    if task_type == "bit_manipulation":
        p, e = predicted_text.removeprefix("0b"), expected_text.removeprefix("0b")
        if not __import__("re").fullmatch(r"[01]{8}", p) or not __import__("re").fullmatch(r"[01]{8}", e):
            return False, "invalid_8bit_format"
        return p == e, "exact_8bit"
    if task_type in {"gravity", "equation_numeric"}:
        if not __import__("re").fullmatch(r"[-+]?\d+(?:\.\d+)?", predicted_text):
            return False, "invalid_numeric_format"
        try:
            ok = math.isclose(float(predicted_text), float(expected_text), rel_tol=1e-2, abs_tol=1e-5)
        except (TypeError, ValueError):
            ok = False
        return ok, "official_math_isclose"
    return predicted_text == expected_text, "exact_text"


### 6.3 vLLM Rollout：保留 Token IDs、Logprob 和 EOS 状态

In [ ]:
def run_vllm_ppo_rollout(prompt_rows: list[dict[str, object]], model_source: str | None, policy_version: str, adapter_path: str | None = None) -> list[dict[str, object]]:
    """Run vLLM only for Rollout; caller persists records before Training."""
    from transformers import AutoTokenizer
    from vllm import LLM, SamplingParams
    model_source = model_source or BASE_MODEL_PATH
    tokenizer = AutoTokenizer.from_pretrained(model_source, trust_remote_code=True, local_files_only=Path(model_source).exists())
    llm = None
    try:
        llm_kwargs = {"model": model_source, "tokenizer": model_source, "tensor_parallel_size": 1, "gpu_memory_utilization": float(os.environ.get("VLLM_GPU_MEMORY_UTILIZATION", "0.80")), "max_model_len": 8192, "enforce_eager": True}
        lora_request = None
        if adapter_path:
            from vllm.lora.request import LoRARequest
            llm_kwargs["enable_lora"] = True
            lora_request = LoRARequest("sft", 1, adapter_path)
        llm = LLM(**llm_kwargs)
        sampling = SamplingParams(temperature=TEMPERATURE, top_p=TOP_P, max_tokens=MAX_NEW_TOKENS, logprobs=1)
        rendered = [tokenizer.apply_chat_template([{"role":"user","content":row["prompt"]}], tokenize=False, add_generation_prompt=True, enable_thinking=True) for row in prompt_rows]
        outputs = llm.generate(rendered, sampling, use_tqdm=True, lora_request=lora_request)
        if len(outputs) != len(prompt_rows):
            raise RuntimeError(f"vLLM rollout 数量错误 expected={len(prompt_rows)} actual={len(outputs)}")
        records = []
        for row, request in zip(prompt_rows, outputs):
            generated = request.outputs[0]
            completion_ids = list(generated.token_ids or [])
            prompt_ids = list(getattr(request, "prompt_token_ids", []) or [])
            if len(completion_ids) != len(generated.logprobs or []):
                raise RuntimeError("vLLM token_ids 与 logprobs 长度不一致")
            truncated = generated.finish_reason == "length"
            correct, verification_method = _verify_rollout(str(row["task_type"]), generated.text, str(row["answer"]), truncated=truncated)
            records.append({"sample_id": str(row["id"]), "task_type": str(row["task_type"]), "prompt": str(row["prompt"]), "expected_answer": str(row["answer"]), "completion": generated.text, "prompt_token_ids": prompt_ids, "completion_token_ids": completion_ids, "response_mask": [0] * len(prompt_ids) + [1] * len(completion_ids), "behavior_logprobs": _chosen_logprob_values(generated), "correct": correct, "verification_method": verification_method, "finish_reason": generated.finish_reason, "truncated": truncated, "policy_version": policy_version, "temperature": TEMPERATURE, "top_p": TOP_P, "source": "vllm_rollout", "verifier_version": OFFICIAL_VERIFIER_VERSION})
        return records
    finally:
        if llm is not None: del llm
        import gc
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()


### 6.4 从 SFT Adapter 加载 Actor-Critic

In [ ]:
def _load_sft_actor_critic(manifest: dict[str, object], mode: str):
    from transformers import AutoModelForCausalLM, BitsAndBytesConfig
    from peft import PeftModel, prepare_model_for_kbit_training
    kwargs = {"trust_remote_code": True, "device_map": "auto"}
    if mode == "nf4_qlora": kwargs["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16)
    elif mode == "bf16_lora": kwargs["torch_dtype"] = torch.bfloat16
    else: raise ValueError(mode)
    base = AutoModelForCausalLM.from_pretrained(manifest["base_model"]["path"], **kwargs)
    if mode == "nf4_qlora": base = prepare_model_for_kbit_training(base, use_gradient_checkpointing=True)
    actor = PeftModel.from_pretrained(base, resolve_manifest_path(str(manifest["adapter_path"])), is_trainable=True)
    return CausalLMValueHead(actor)


### 6.5 把 Rollout 转成 PPO 时间轴张量

In [ ]:
def build_ppo_rollout_from_records(records: list[dict[str, object]], actor_critic: nn.Module, pad_token_id: int = 0) -> PPORollout:
    if not records: raise ValueError("empty PPO rollout")
    sequences, prompt_lengths, completion_lengths, old_logprob_rows = [], [], [], []
    rewards_correct = []
    for record in records:
        prompt_ids, completion_ids = list(record["prompt_token_ids"]), list(record["completion_token_ids"])
        if not prompt_ids: raise ValueError("PPO rollout missing prompt_token_ids")
        if len(completion_ids) != len(record["behavior_logprobs"]): raise ValueError("PPO behavior Logprob alignment error")
        sequences.append(prompt_ids + completion_ids); prompt_lengths.append(len(prompt_ids)); completion_lengths.append(len(completion_ids)); old_logprob_rows.append([float(v) for v in record["behavior_logprobs"]]); rewards_correct.append(bool(record["correct"]))
    max_len = max(map(len, sequences)); batch = len(sequences)
    device = next(actor_critic.parameters()).device
    input_ids = torch.full((batch, max_len), int(pad_token_id), dtype=torch.long, device=device)
    attention_mask = torch.zeros((batch, max_len), dtype=torch.long, device=device)
    response_mask = torch.zeros((batch, max_len), dtype=torch.long, device=device)
    old_logprobs = torch.zeros((batch, max_len), dtype=torch.float32, device=device)
    dones = torch.zeros((batch, max_len), dtype=torch.float32, device=device)
    for row, (sequence, prompt_len, completion_len, old_values, record) in enumerate(zip(sequences, prompt_lengths, completion_lengths, old_logprob_rows, records)):
        input_ids[row, :len(sequence)] = torch.tensor(sequence, dtype=torch.long, device=device); attention_mask[row, :len(sequence)] = 1; response_mask[row, prompt_len:len(sequence)] = 1
        if old_values: old_logprobs[row, prompt_len:prompt_len + completion_len] = torch.tensor(old_values, dtype=torch.float32, device=device)
        if record.get("finish_reason") == "stop" and not record.get("truncated") and completion_len: dones[row, prompt_len + completion_len - 1] = 1
    rewards = outcome_reward(torch.tensor(rewards_correct, device=device), response_mask, str(records[0]["task_type"]))
    with torch.no_grad():
        actor_critic.eval(); old_values = actor_critic(input_ids, attention_mask)["values"].detach()
    rollout = PPORollout(input_ids, attention_mask, response_mask, old_logprobs, old_values, rewards, dones, str(records[0].get("policy_version", "unknown")), OFFICIAL_VERIFIER_VERSION, str(records[0].get("finish_reason", "unknown")), bool(any(record.get("truncated") for record in records)), str(records[0]["task_type"]), str(records[0]["sample_id"]))
    validate_rollout_alignment(rollout)
    return rollout


### 6.6 保存 PPO Adapter、Value Head 与 Manifest

In [ ]:
def _save_ppo_update(model, logs: list[dict[str, float]], sft_manifest_path: Path, mode: str, parent_manifest: dict[str, object] | None = None) -> dict[str, object]:
    output_dir = CHECKPOINT_ROOT / f"ppo_{mode}"
    output_dir.mkdir(parents=True, exist_ok=True)
    model.base_model.save_pretrained(output_dir)
    torch.save(model.value_head.state_dict(), output_dir / "value_head.pt")
    parent_manifest = parent_manifest or read_sft_manifest(sft_manifest_path)
    ppo_manifest = {"schema_version":"nemotron-ppo-checkpoint/v1", "status":"pending_reload", "parent_sft_manifest":str(sft_manifest_path.relative_to(PROJECT_ROOT)), "parent_sft_adapter":str(parent_manifest["adapter_path"]), "base_model":parent_manifest["base_model"], "tokenizer_path":parent_manifest.get("tokenizer_path"), "adapter_path":str(output_dir.relative_to(PROJECT_ROOT)), "value_head_path":str((output_dir / "value_head.pt").relative_to(PROJECT_ROOT)), "mode":mode, "policy_version":f"ppo:{mode}:seed{SEED}", "seed":SEED, "profile":TRAINING_PROFILE}
    (output_dir / "ppo_manifest.json").write_text(json.dumps(ppo_manifest, ensure_ascii=False, indent=2), encoding="utf-8")
    return {"status":"artifact_saved", "updates":len(logs), "logs":logs, "adapter_path":str(output_dir), "value_head_path":str(output_dir / "value_head.pt"), "manifest_path":str(output_dir / "ppo_manifest.json")}


def _reload_ppo_artifact(parent_manifest: dict[str, object], artifact: dict[str, object], mode: str) -> dict[str, object]:
    """Reload Adapter + Value Head from disk before reporting PPO completion."""
    model = None
    try:
        model = _load_sft_actor_critic(parent_manifest, mode, value_head_path=Path(artifact["value_head_path"]))
        return {"status": "passed", "value_head_device": str(model.value_head.weight.device), "value_head_dtype": str(model.value_head.weight.dtype)}
    except torch.cuda.OutOfMemoryError as exc:
        return {"status": "oom", "error": repr(exc)}
    except Exception as exc:
        return {"status": "failed", "error": repr(exc)}
    finally:
        if model is not None: del model
        import gc
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()


### 6.7 PPO Update 调度

In [ ]:
def run_ppo_training(rollouts: list[PPORollout], sft_manifest_path: Path = SFT_MANIFEST_PATH, mode: str = "nf4_qlora", actor_critic: nn.Module | None = None) -> dict[str, object]:
    if not RUN_PPO_TRAINING:
        return {"status": "skipped_by_default", "stage": "ppo_training", "reason": "RUN_PPO_TRAINING=False"}
    if not rollouts:
        return {"status": "blocked_empty_rollout", "stage": "ppo_training"}
    if not torch.cuda.is_available():
        return {"status": "blocked_no_cuda", "stage": "ppo_training"}
    started = time.perf_counter()
    model = None
    try:
        manifest = read_sft_manifest(sft_manifest_path)
        model = actor_critic or _load_sft_actor_critic(manifest, mode)
        optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=LEARNING_RATE)
        logs = [ppo_update(model, optimizer, rollout) for rollout in rollouts]
        artifact = _save_ppo_update(model, logs, sft_manifest_path, mode, parent_manifest=manifest)
        del model
        model = None
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        reload_check = _reload_ppo_artifact(manifest, artifact, mode)
        if reload_check.get("status") != "passed":
            return {"status": "failed_adapter_reload", "stage": "ppo_training", "artifact": artifact, "adapter_reload": reload_check}
        artifact["status"] = "completed"
        artifact.update({"stage": "ppo_training", "training_time_seconds": time.perf_counter() - started, "adapter_reload": reload_check, "gpu_device": torch.cuda.get_device_name(0), "peak_memory_allocated_mib": torch.cuda.max_memory_allocated() / 2**20, "real_model_forward_backward": True})
        return artifact
    except torch.cuda.OutOfMemoryError as exc:
        return {"status": "oom", "stage": "ppo_training", "error": repr(exc), "wall_time_seconds": time.perf_counter() - started}
    except Exception as exc:
        return {"status": "failed", "stage": "ppo_training", "error": repr(exc), "wall_time_seconds": time.perf_counter() - started}
    finally:
        if model is not None: del model
        import gc
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()


### 6.8 选择 Train Smoke Prompt

In [ ]:
def select_ppo_smoke_prompts(per_task: int = SMOKE_PROMPTS_PER_TASK) -> list[dict[str, object]]:
    if not TRAIN_PATH.exists(): raise FileNotFoundError("缺少 Train 划分")
    frame = pd.read_csv(TRAIN_PATH, dtype=str, keep_default_na=False)
    selected = []
    for task, offset in (("equation_numeric", 1), ("bit_manipulation", 2)):
        pool = frame[frame["task_type"] == task]
        for row in pool.sample(n=min(per_task, len(pool)), random_state=SEED + offset).itertuples(index=False):
            selected.append({"id":str(row.id), "task_type":str(row.task_type), "prompt":str(row.prompt), "answer":str(row.answer), "source_split":"train"})
    return selected


### 6.9 PPO Smoke Pipeline：Rollout → GAE → Update

In [ ]:
def _write_ppo_jsonl(path: Path, rows: list[dict[str, object]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text("\n".join(json.dumps(row, ensure_ascii=False) for row in rows) + ("\n" if rows else ""), encoding="utf-8")


def _read_ppo_jsonl(path: Path) -> list[dict[str, object]]:
    if not path.exists(): raise FileNotFoundError(f"阶段产物不存在：{path}")
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]


def _ppo_preflight(stage: str) -> dict[str, object]:
    if not torch.cuda.is_available(): return {"status":"blocked_no_cuda", "stage":stage}
    free, total = torch.cuda.mem_get_info()
    result = {"status":"ready", "stage":stage, "device":torch.cuda.get_device_name(0), "free_memory_mib":free/2**20, "total_memory_mib":total/2**20}
    RESULT_DIR.mkdir(parents=True, exist_ok=True)
    (RESULT_DIR / f"{stage}_preflight.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
    return result


def run_ppo_rollout_phase() -> dict[str, object]:
    if not RUN_PPO_ROLLOUT: return {"status":"skipped_by_default", "stage":"ppo_rollout"}
    preflight = _ppo_preflight("ppo_rollout")
    if preflight["status"] != "ready": return preflight
    started = time.perf_counter()
    try:
        manifest = read_sft_manifest()
        prompts = select_ppo_smoke_prompts()
        records = run_vllm_ppo_rollout(prompts, manifest["base_model"]["path"], str(manifest.get("policy_version", "sft_policy")), adapter_path=resolve_manifest_path(str(manifest["adapter_path"])))
        _write_ppo_jsonl(PPO_ROLLOUT_PATH, records)
        payload = {"status":"completed", "stage":"ppo_rollout", "records":len(records), "prompts":len(prompts), "sft_manifest":str(SFT_MANIFEST_PATH), "rollout_path":str(PPO_ROLLOUT_PATH), "wall_time_seconds":time.perf_counter()-started, "gpu_device":torch.cuda.get_device_name(0), "peak_memory_allocated_mib":torch.cuda.max_memory_allocated()/2**20}
        PPO_ROLLOUT_MANIFEST_PATH.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
        return payload
    except torch.cuda.OutOfMemoryError as exc:
        return {"status":"oom", "stage":"ppo_rollout", "error":repr(exc), "wall_time_seconds":time.perf_counter()-started}
    except Exception as exc:
        return {"status":"failed", "stage":"ppo_rollout", "error":repr(exc), "wall_time_seconds":time.perf_counter()-started}


def run_ppo_training_phase(mode: str = "nf4_qlora") -> dict[str, object]:
    if not RUN_PPO_TRAINING: return {"status":"skipped_by_default", "stage":"ppo_training"}
    started = time.perf_counter()
    try:
        records = _read_ppo_jsonl(PPO_ROLLOUT_PATH)
        manifest = read_sft_manifest()
        tokenizer_id = manifest.get("tokenizer_path", manifest["base_model"]["path"])
        from transformers import AutoTokenizer
        tokenizer = AutoTokenizer.from_pretrained(tokenizer_id, trust_remote_code=True, local_files_only=Path(tokenizer_id).exists())
        model = _load_sft_actor_critic(manifest, mode)
        pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
        rollout = build_ppo_rollout_from_records(records, model, pad_token_id=int(pad_id))
        del model
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        result = run_ppo_training([rollout], SFT_MANIFEST_PATH, mode)
        payload = {"status": result.get("status", "failed"), "stage":"ppo_training", "rollout_path":str(PPO_ROLLOUT_PATH), "training":result, "wall_time_seconds":time.perf_counter()-started}
        PPO_TRAINING_MANIFEST_PATH.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
        return payload
    except torch.cuda.OutOfMemoryError as exc:
        payload = {"status":"oom", "stage":"ppo_training", "error":repr(exc), "wall_time_seconds":time.perf_counter()-started}
        PPO_TRAINING_MANIFEST_PATH.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
        return payload
    except Exception as exc:
        payload = {"status":"failed", "stage":"ppo_training", "error":repr(exc), "wall_time_seconds":time.perf_counter()-started}
        PPO_TRAINING_MANIFEST_PATH.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
        return payload


def run_ppo_smoke_pipeline(mode: str = "nf4_qlora") -> dict[str, object]:
    results = {}
    if RUN_PPO_ROLLOUT:
        results["rollout"] = run_ppo_rollout_phase()
        if results["rollout"].get("status") != "completed" and RUN_PPO_TRAINING:
            return {"status":"blocked_rollout_failed", **results}
    if RUN_PPO_TRAINING:
        results["training"] = run_ppo_training_phase(mode)
        if results["training"].get("status") != "completed":
            return {"status":results["training"].get("status", "failed"), **results}
    return {"status":"completed", **results} if results else {"status":"skipped_by_default", "stage":"ppo"}


## 7. CPU Smoke Test

Smoke Test 使用 Tiny Actor-Critic 和人工构造的张量，只验证 GAE 终止状态、Mask、Ratio、Clip Fraction、Value/Entropy/KL 和反向传播。它不是 Qwen3 的 PPO 结果，也不证明真实 Reward 有效。

In [ ]:
rollout = build_cpu_rollout()
action_rewards, action_old_values, action_dones, action_mask = align_ppo_action_axis(
    rollout.rewards, rollout.old_values, rollout.dones, rollout.response_mask
)
assert action_rewards.shape == (rollout.rewards.shape[0], rollout.rewards.shape[1] - 1)
assert torch.equal(action_mask, rollout.response_mask[:, 1:].float())
# Short-sequence alignment: token at position 2 uses state/value at position 1.
manual_values = torch.tensor([[99.0, 10.0, 20.0, 30.0]])
manual_rewards = torch.tensor([[0.0, 0.0, 0.0, 1.0]])
manual_dones = torch.tensor([[0.0, 0.0, 0.0, 1.0]])
manual_mask = torch.tensor([[0.0, 0.0, 1.0, 1.0]])
manual_r, manual_v, manual_d, manual_m = align_ppo_action_axis(
    manual_rewards, manual_values, manual_dones, manual_mask
)
assert torch.equal(manual_v, torch.tensor([[99.0, 10.0, 20.0]]))
assert torch.equal(manual_r, torch.tensor([[0.0, 0.0, 1.0]]))
assert torch.equal(manual_m, torch.tensor([[0.0, 1.0, 1.0]]))
advantages, returns = compute_gae(action_rewards, action_old_values, action_dones, action_mask)
assert advantages.shape == action_rewards.shape
assert returns.shape == action_rewards.shape
assert torch.all(advantages[:, :1] == 0)
assert torch.all(returns[:, :1] == 0)

model = TinyActorCritic()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-2)
logs = ppo_update(model, optimizer, rollout)
assert all(math.isfinite(value) for value in logs.values())
assert logs["clip_fraction"] >= 0.0
assert any(parameter.grad is not None for parameter in model.parameters())

# A terminal reward must not bootstrap through the next state.
terminal_rewards = torch.tensor([[0.0, 1.0]])
terminal_values = torch.tensor([[0.3, 0.4]])
terminal_dones = torch.tensor([[0.0, 1.0]])
terminal_mask = torch.ones_like(terminal_rewards)
terminal_adv, _ = compute_gae(terminal_rewards, terminal_values, terminal_dones, terminal_mask, gamma=0.9, lam=0.95)
expected_last = terminal_rewards[0, 1] - terminal_values[0, 1]
assert torch.isclose(terminal_adv[0, 1], expected_last)
# Padding after EOS must not propagate into the preceding response token.
padded_rewards = torch.tensor([[0.0, 1.0, 0.0, 0.0]])
padded_values = torch.tensor([[0.3, 0.4, 9.0, 9.0]])
padded_dones = torch.tensor([[0.0, 1.0, 0.0, 0.0]])
padded_mask = torch.tensor([[1.0, 1.0, 0.0, 0.0]])
padded_adv, padded_ret = compute_gae(padded_rewards, padded_values, padded_dones, padded_mask)
assert torch.all(padded_adv[:, 2:] == 0) and torch.all(padded_ret[:, 2:] == 0)

ppo_cpu_report = {
    "status": "passed",
    "gae_shape_and_mask": True,
    "value_action_alignment": True,
    "verifier_version": OFFICIAL_VERIFIER_VERSION,
    "terminal_bootstrap_disabled": True,
    "ppo_loss_finite": True,
    "policy_gradient": True,
    "clip_fraction": logs["clip_fraction"],
    "approx_kl": logs["approx_kl"],
    "gpu_rollout": "not_run",
    "gpu_training": "not_run",
}
print(json.dumps(ppo_cpu_report, ensure_ascii=False, indent=2))

## 本节总结与限制

已完成标准 PPO 的核心数值实现和 CPU Smoke Test。当前没有真实 Qwen3 Actor、Critic、vLLM Rollout 或 GPU 更新结果。明天先用少量 Equation/Bit Rollout 检查行为 Logprobs、Response Mask、EOS 和 Critic Value 对齐，再运行一轮 PPO；若 Reward 上升但独立 Accuracy 不升，结论应记录为训练诊断而不是能力提升。

## 8. SFT Adapter 交接检查

PPO 只从 Notebook 03 的 SFT Manifest 加载 Base Model 和 PEFT Adapter；Value Head 单独保存。当前 CPU 阶段不加载模型。


In [ ]:
ppo_manifest_status = "available" if SFT_MANIFEST_PATH.exists() else "not_run_sft_manifest_missing"
if not (RUN_PPO_ROLLOUT or RUN_PPO_TRAINING):
    assert RUN_PPO is False
ppo_checkpoint_report = {"status":"passed", "smoke_prompts_per_task": SMOKE_PROMPTS_PER_TASK, "temperature": TEMPERATURE, "top_p": TOP_P, "sft_manifest": ppo_manifest_status, "gae_padding_mask_test": True, "value_action_alignment": True, "verifier_version": OFFICIAL_VERIFIER_VERSION, "gpu":"not_run"}
(RESULT_DIR / "cpu_ppo_smoke_report.json").parent.mkdir(parents=True, exist_ok=True)
(RESULT_DIR / "cpu_ppo_smoke_report.json").write_text(json.dumps(ppo_checkpoint_report, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(ppo_checkpoint_report, ensure_ascii=False, indent=2))
